<p align="center"><a href="https://www.plus2net.com/python/file-truncate.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python file.truncate(): Resizing Files & Managing Disk Usage Practice

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/file-truncate.php](https://www.plus2net.com/python/file-truncate.php)

Run cells in order in Google Colab. All file resizing concepts, explicit byte limits vs implicit cursor truncation, required write access modes (`'r+'`), zero-padding behavior when extending files, clearing files safely in-place, and self-checking assertions are embedded directly in this notebook. Save a copy in Google Drive to keep your edits.

### 1. Setup: Creating Sample Files for Truncation
Generate test files with known byte sizes.

In [ ]:
import os

sample_payload = b"0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"  # 36 bytes
with open("demo_truncate.bin", "wb") as f:
    f.write(sample_payload)

print("Created demo_truncate.bin, initial size:", os.path.getsize("demo_truncate.bin"), "bytes")


### 2. Truncating to an Explicit Byte Size in Mode 'r+'
Resizing to a smaller size truncates the file and discards trailing bytes.

In [ ]:
with open("demo_truncate.bin", "r+b") as stream:
    new_size = stream.truncate(15)  # Shrink to 15 bytes
    print("Truncate returned new size:", new_size)

with open("demo_truncate.bin", "rb") as stream:
    print("Content after truncation: ", stream.read())  # b'0123456789ABCDE'
    print("Current file size on disk:", os.path.getsize("demo_truncate.bin"))


### 3. Implicit Truncation at Current Cursor Offset (size=None)
When called without arguments, `truncate()` resizes the file at the current `tell()` position.

In [ ]:
with open("demo_truncate.bin", "r+b") as stream:
    stream.seek(10)     # Move pointer to byte 10
    stream.truncate()   # Truncate at current position

with open("demo_truncate.bin", "rb") as stream:
    print("Content after seek(10) + truncate():", stream.read())  # b'0123456789'
    print("File size now:                     ", os.path.getsize("demo_truncate.bin"))


### 4. Clearing a File Completely In-Place with truncate(0)
Clear an existing file to 0 bytes without deleting and recreating it.

In [ ]:
with open("demo_truncate.bin", "r+b") as stream:
    stream.truncate(0)

print("Size after truncate(0):", os.path.getsize("demo_truncate.bin"), "bytes")


### 5. Extending a File with Sparse Zero-Byte Padding
Truncating to a larger size extends the file, padding the new area with null bytes (`\x00`).

In [ ]:
with open("demo_truncate.bin", "r+b") as stream:
    stream.truncate(20)  # Extend to 20 bytes

with open("demo_truncate.bin", "rb") as stream:
    data = stream.read()
    print("Padded binary payload:", data)
    print("All padding is null:   ", all(b == 0 for b in data))


### 6. Read-Only Mode Trap Demonstration
Attempting to call `truncate()` on a read-only stream raises `io.UnsupportedOperation`.

In [ ]:
import io

with open("demo_truncate.bin", "rb") as stream:
    try:
        stream.truncate(5)
    except io.UnsupportedOperation as err:
        print("Caught expected read-only error:", err)


### 7. Self-Checking Verification Suite
Run unit assertions verifying file shrinking, clearing, and error handling.

In [ ]:
class TruncateVerifier:
    @staticmethod
    def shrink(path: str, size: int) -> int:
        with open(path, "r+b") as stream:
            stream.truncate(size)
        return os.path.getsize(path)

# Verification assertions
v = TruncateVerifier()
test_file = "verify_truncate.bin"
with open(test_file, "wb") as f:
    f.write(b"ABCDEFGHIJKLMNOPQRSTUVWXYZ")  # 26 bytes

assert v.shrink(test_file, 10) == 10
with open(test_file, "rb") as f:
    assert f.read() == b"ABCDEFGHIJ"

with open(test_file, "r+b") as f:
    f.seek(4)
    f.truncate()
assert os.path.getsize(test_file) == 4

with open(test_file, "r+b") as f:
    f.truncate(0)
assert os.path.getsize(test_file) == 0

print("All file.truncate() unit assertions passed successfully! 100% verified.")
